In [3]:
import pandas as pd
import numpy as np
import warnings
from sklearn.exceptions import UndefinedMetricWarning
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedShuffleSplit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

warnings.filterwarnings("ignore", category=UndefinedMetricWarning)

df = pd.read_csv("emails.csv")

X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['text'], df['spam'], test_size=0.2, random_state=42
)

tfidf = TfidfVectorizer(max_features=1000, ngram_range=(1, 1))
X_train = tfidf.fit_transform(X_train_text)
X_test = tfidf.transform(X_test_text)

models = {
    "MultinomialNB": MultinomialNB(),
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=42),
    "RandomForest": RandomForestClassifier(n_jobs=-1, random_state=42),
    "HistGradientBoosting": HistGradientBoostingClassifier(random_state=42),
    "LinearSVC": LinearSVC(random_state=42)
}

model_predictions_test = {}
print("\n--- Individual Model Evaluations ---\n")

for name, model in models.items():
    if name == "HistGradientBoosting":
        model.fit(X_train.toarray(), y_train)
        preds = model.predict(X_test.toarray())
    else:
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
    
    model_predictions_test[name] = preds
    acc = accuracy_score(y_test, preds)
    report = classification_report(y_test, preds)
    print(f"--- {name} ---")
    print(f"Accuracy: {acc:.4f}")
    print(report)

print("\n--- Ensemble (Majority Vote) - Test Set Evaluation ---")
pred_array = np.array(list(model_predictions_test.values()))
ensemble_preds = np.round(np.mean(pred_array, axis=0)).astype(int)
ensemble_accuracy = accuracy_score(y_test, ensemble_preds)
ensemble_report = classification_report(y_test, ensemble_preds)
print(f"Ensemble Accuracy: {ensemble_accuracy:.4f}")
print(ensemble_report)

def predict_spam(message, model, dense=False):
    vec = tfidf.transform([message])
    return model.predict(vec.toarray() if dense else vec)[0]

message = input("\nEnter a message to classify:\n\n")

model_votes = []
print("\n--- Model Predictions ---")
for name, model in models.items():
    prediction = predict_spam(message, model, dense=(name == "HistGradientBoosting"))
    model_votes.append(prediction)
    print(f"{name}: {'Spam' if prediction == 1 else 'Not Spam'}")

final_prediction = "Spam" if model_votes.count(1) > model_votes.count(0) else "Not Spam"
print(f"\nFinal Ensemble Prediction: {final_prediction}")

print("\n--- Cross-Validation (HistGradientBoosting) ---")
sss = StratifiedShuffleSplit(n_splits=3, test_size=0.2, random_state=42)
dense_X = tfidf.transform(df['text']).toarray()
scores = cross_val_score(models["HistGradientBoosting"], dense_X, df['spam'], cv=sss)
print(f"Cross-validation scores: {scores}")
print(f"Mean CV score: {scores.mean():.4f}")




--- Individual Model Evaluations ---

--- MultinomialNB ---
Accuracy: 0.9686
              precision    recall  f1-score   support

           0       0.97      0.99      0.98       856
           1       0.97      0.90      0.94       290

    accuracy                           0.97      1146
   macro avg       0.97      0.95      0.96      1146
weighted avg       0.97      0.97      0.97      1146

--- LogisticRegression ---
Accuracy: 0.9791
              precision    recall  f1-score   support

           0       0.98      1.00      0.99       856
           1       0.99      0.93      0.96       290

    accuracy                           0.98      1146
   macro avg       0.98      0.96      0.97      1146
weighted avg       0.98      0.98      0.98      1146

--- RandomForest ---
Accuracy: 0.9747
              precision    recall  f1-score   support

           0       0.97      1.00      0.98       856
           1       0.99      0.91      0.95       290

    accuracy          


Enter a message to classify:

 tell these cam sluts what to do  to be removed please click here or simply respond to this email . your address will be removed and blocked from ever being added again .  please scroll down to the bottom of this email for more details .  all these shows are live right now !  ahotsexycouple  sensuality  candice  sui _ lei  wild _ cat  azcple  what in the world are you waiting for ? click now and tell any of these women what to do for you live on camera . don ' t be shy , just signup for free and tell them what you want and you will get .  click here for the free live show !  this is our central mailing for all of our affiliate sites . if you have a question on how you got on , please email us and we will be glad to help .  the fastest way to get off our list is to click this link . if you do not have access to it , please respond to this email .  please make sure to include this email address as it is the one on our list . zzzz @ example . com



--- Model Predictions ---
MultinomialNB: Spam
LogisticRegression: Spam
RandomForest: Spam
HistGradientBoosting: Spam
LinearSVC: Spam

Final Ensemble Prediction: Spam

--- Cross-Validation (HistGradientBoosting) ---
Cross-validation scores: [0.97818499 0.991274   0.98254799]
Mean CV score: 0.9840
